# 实验二：算子原型——CANN算子接口理解与跨生态洞察

## 小节概述

理解 CANN 算子接口定义规范，并与 PyTorch/CUDA 等主流生态进行接口设计对比。

## 教程具体内容

以下内容围绕实验背景、任务准备、关键步骤和实验总结展开。


建议学时：2学时

## 实验任务

### 任务描述

---

原实验设计：

在华为云沙箱完成算子接口分析、ONNX导出、ATC转换和后端表示观察。

修改后的设计：

本实验在CANNLAB平台完成，重点在于：

- 理解CANN算子接口定义规范

- 对比CANN与PyTorch/CUDA的接口设计差异

- 掌握OpDef、输入输出约束、Shape/Type推导规则

- 为后续Host/Kernel开发建立接口认知基础

修改原因：

算子接口理解是算子开发的前置知识，不需要真实NPU硬件，适合在CANNLAB环境完成。学生可以专注于接口规范学习，而不被环境问题干扰。

---

### 学习目标

小组分工：

- 开源软件开发工程师：主导算子接口定义分析、OpDef规范整理

- 开源社区布道师：负责跨生态接口对比、约束表达方式提取

- 开源合规与安全工程师：审查接口定义的一致性和可复验性

学习资源路径：

- CANNLAB平台：使用预置的算子样例和文档资源

- 官方学习仓库：https://gitcode.com/cann/cann-learning-hub

- 接口定义文档：CANN自定义算子开发文档中的OpDef章节

---

## 任务准备

### 前置知识

本实验需要提前学习以下相关知识：

- CANN异构计算架构基础知识
- Python编程基础
- Linux命令行操作基础

### 实验环境准备

本实验需要在以下环境中进行：

- **CANNLAB平台**：访问CANNLAB在线实验平台，获取预配置的算子开发环境

## 任务实施

### 实验要点

- 步骤一：在CANNLAB分析基础算子接口定义
- 步骤二：对比CANN与PyTorch的接口设计
- 步骤三：理解CANN的输入输出约束机制
- 步骤四：完成基础算子的接口定义说明文档
- 步骤五：设计异常场景测试用例

### 关键步骤

步骤一：在CANNLAB分析基础算子接口定义

以sinh算子为例，分析CANN的OpDef定义：

In [ ]:
# 在当前 CANN 安装目录中查找 Sinh 的实现或注册文件
import os
from pathlib import Path
import subprocess

ascend_home_value = os.environ.get("ASCEND_HOME_PATH") or os.environ.get("ASCEND_HOME")
if not ascend_home_value:
    raise RuntimeError("ASCEND_HOME_PATH 或 ASCEND_HOME 未设置，请先加载 CANN 环境。")

ascend_home = Path(ascend_home_value).expanduser().resolve()
if not ascend_home.is_dir():
    raise RuntimeError(f"CANN 安装目录不存在: {ascend_home}")

print(f"CANN 安装目录: {ascend_home}")
search = subprocess.run(
    ["find", str(ascend_home), "-type", "f", "-iname", "*sinh*"],
    check=False,
    capture_output=True,
    text=True,
)
matches = [Path(line) for line in search.stdout.splitlines() if line.strip()][:10]

if not matches:
    print("当前 CANN 包未暴露独立的 Sinh 定义文件；后续使用接口表继续完成原型分析。")
else:
    print("找到以下 Sinh 相关文件（最多显示 10 个）:")
    for match in matches:
        print(f"  {match}")

    preview = matches[0]
    print(f"\n=== {preview.name} 前 50 行 ===")
    try:
        lines = preview.read_text(encoding="utf-8", errors="replace").splitlines()
        print("\n".join(lines[:50]))
    except OSError as exc:
        print(f"文件存在但无法读取: {exc}")


步骤二：对比CANN与PyTorch的接口设计
在CANNLAB中编写对比分析代码：

In [ ]:
import torch

# PyTorch中的sinh算子调用
x_torch = torch.randn(2, 256, dtype=torch.float32)
y_torch = torch.sinh(x_torch)

print("=== PyTorch sinh接口 ===")
print("Input shape:", x_torch.shape, "dtype:", x_torch.dtype)
print("Output shape:", y_torch.shape, "dtype:", y_torch.dtype)
print("接口特点: 自动类型推导，无需显式指定输出类型")

# CANN中的算子定义规范（概念性展示）
cann_op_def = {
    "op_name": "Sinh",
    "inputs": [{
        "name": "x",
        "dtype": ["float16", "float32"],
        "format": ["ND"]
    }],
    "outputs": [{
        "name": "y",
        "dtype": "same_as_input",
        "format": "same_as_input"
    }],
    "attrs": [],
    "shape_infer": "output_shape == input_shape",
    "type_infer": "output_dtype == input_dtype"
}

print("\n=== CANN sinh接口定义 ===")
print("接口特点: 显式类型约束，需要定义支持的dtype列表")
print("OpDef:", cann_op_def)

步骤三：理解CANN的输入输出约束机制
分析CANN算子的约束表达方式：

In [ ]:
# CANN算子约束分析
constraints = {
    "数据类型约束": {
        "CANN": "显式枚举支持的dtype列表，如['float16', 'float32']",
        "PyTorch": "隐式支持，根据输入自动推导",
        "差异": "CANN更严格，需要在定义时明确约束范围"
    },
    "数据格式约束": {
        "CANN": "需要指定张量排布格式，如ND、NCHW、NC1HWC0",
        "PyTorch": "通常不关心底层排布，由框架自动处理",
        "差异": "CANN面向硬件优化，需要考虑NPU内存排布"
    },
    "Shape推导": {
        "CANN": "需要实现InferShape函数，显式计算输出shape",
        "PyTorch": "框架自动推导，用户无需关心",
        "差异": "CANN需要开发者参与shape推导逻辑"
    },
    "Type推导": {
        "CANN": "需要实现InferType函数，处理类型映射",
        "PyTorch": "自动类型提升和推导",
        "差异": "CANN要求显式类型推导规则"
    }
}

for key, value in constraints.items():
    print(f"\n=== {key} ===")
    for k, v in value.items():
        print(f"{k}: {v}")

步骤四：完成基础算子的接口定义说明文档
为后续开发准备接口定义文档：

## Sinh算子接口定义说明

### 1. 算子名称
- CANN: Sinh
- PyTorch: torch.sinh
- ONNX: Sinh (Opset 13+)

### 2. 输入描述
| 参数名 | 类型 | 格式 | 约束 |
|--------|------|------|------|
| x | float16/float32 | ND | 任意维度张量 |

### 3. 输出描述

| 参数名 | 类型 | 格式 | 推导规则 |
|--------|------|------|------|
| y | same_as_x | same_as_x | Output.shape=input.shape |

### 4. 属性

无

### 5. Shape推导规则

output_shape[i] = input_shape[i] for all i

### 6. Type推导规则

output_dtype = input_dtype

### 7. 异常处理

- 空张量：返回空张量

- 不支持的dtype：返回ACL_ERROR_INVALID_PARAM

- 维度过大：根据硬件限制返回错误

步骤五：设计异常场景测试用例

In [ ]:
# 异常场景设计
test_cases = [
    {
        "场景": "空张量输入",
        "输入": torch.tensor([], dtype=torch.float32),
        "预期": "返回空张量",
        "类型": "边界条件"
    },
    {
        "场景": "不支持的数据类型",
        "输入": torch.randint(0, 100, (2, 3), dtype=torch.int64),
        "预期": "类型错误或按算子规范转换",
        "类型": "类型约束"
    },
    {
        "场景": "超大维度张量",
        "输入": "shape=(1024, 1024, 1024), dtype=float32（仅描述，不分配内存）",
        "预期": "检查实现支持上限；超限时返回明确错误",
        "类型": "资源限制"
    }
]

for i, case in enumerate(test_cases, 1):
    print(f"\n测试用例{i}: {case['场景']}")
    print(f"类型: {case['类型']}")
    print(f"预期结果: {case['预期']}")


## 任务拓展

《基础算子接口定义说明.md》：内容必须包含CANN OpDef关键字段说明表、输入输出与属性约束表、Shape/Type推导规则说明、异常输入处理规则表。

《跨生态接口对比分析.md》：内容必须包含CANN与PyTorch/CUDA的接口对比表、约束表达方式差异分析、接口设计哲学对比（硬件友好 vs 框架友好）。

## 实验总结

通过本次实验，完成了以下关键学习目标：

- 理解了CANN算子接口定义规范
- 完成了CANN与PyTorch/CUDA的接口对比分析
- 掌握了OpDef、输入输出约束、Shape/Type推导规则

## 课后实践

请选择一个基础算子，整理接口定义说明，并完成 CANN 与主流生态的接口约束对比表。

## 参考答案

运行下面的代码单元查看以 Sinh 算子为例的参考答案。


In [ ]:
!cat ./answer/01.02_answer.md